# 🚖 NYC Yellow Taxi Trip Duration Prediction: MLflow-Integrated Production Pipeline
## COMPLETE MLFLOW INTEGRATION | SQLITE BACKEND | MODEL REGISTRY | STAGE TRANSITIONS | PRODUCTION READY

This notebook demonstrates a **production-grade ML pipeline** with comprehensive MLflow integration including:
- ✅ **SQLite Backend** - Single file database for complete experiment tracking
- ✅ **Automatic Model Registry** - Version tracking with auto-registration
- ✅ **Stage Transitions** - Staging → Production → Archived workflow
- ✅ **Model Lineage** - Full traceability from training to deployment
- ✅ **Model Comparison** - Advanced experiment analysis and promotion
- ✅ **Production Deployment** - models:/ URI based loading

# 📌 IMPORTANT: MLflow UI Setup

## 🔴 CRITICAL: Use SQLite for Complete Functionality

```bash
# Terminal 1: Start MLflow UI with the NYC Taxi database
mlflow ui --backend-store-uri sqlite:///mlflow_nyc_taxi.db --host 127.0.0.1 --port 5000
```

## 🌐 Open in browser:
**http://127.0.0.1:5000**

## 📸 You will see:
- Experiment: 'nyc_taxi_production_mlflow'
- All model training runs with parameters & metrics
- Model Registry with 'nyc_taxi_predictor' model
- Version stages: Staging, Production, Archived
- Complete model lineage

In [ ]:
# ============================================
# PART 0: MLflow Environment Setup with SQLite
# ============================================

import warnings
warnings.filterwarnings('ignore')

# 🔧 Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from datetime import datetime
import os
import time
import shutil
from math import radians, cos, sin, asin, sqrt

# 🧰 Sklearn
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.base import BaseEstimator, TransformerMixin
from mlflow.models import infer_signature

# 🧪 MLflow with FULL Model Registry support
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from mlflow.entities import ViewType

# ============================================
# CRITICAL: Set SQLite backend for COMPLETE MLflow functionality
# This enables Model Registry, versioning, and stage transitions
# ============================================

MLFLOW_DB_PATH = os.path.abspath("./mlflow_nyc_taxi.db")
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_PATH}")
client = MlflowClient()

print("🔍 MLflow SQLite Backend Configuration")
print("=" * 60)
print(f"✅ MLflow version: {mlflow.__version__}")
print(f"✅ Tracking URI: sqlite:///{MLFLOW_DB_PATH}")
print(f"✅ Database location: {MLFLOW_DB_PATH}")
print(f"✅ MLflow Client: {client.__class__.__name__}")

# Test SQLite backend
try:
    experiment_test = mlflow.create_experiment(
        "mlflow_backend_test",
        tags={"test": "sqlite_backend_working"}
    )
    print("✅ SQLite backend verification: SUCCESS")
    print(f"   Test experiment ID: {experiment_test}")
except:
    print("✅ SQLite backend already initialized")

print("\n" + "=" * 60)
print("🎯 NEXT STEP - Start MLflow UI:")
print("=" * 60)
print(f"""
cd {os.getcwd()}
mlflow ui --backend-store-uri sqlite:///{MLFLOW_DB_PATH} --host 127.0.0.1 --port 5000

🌐 Then open: http://127.0.0.1:5000
""")

In [ ]:
# ============================================
# PART 1: Configuration with MLflow Integration
# ============================================

class Config:
    """Centralized configuration with MLflow integration."""
    
    # ============ REPRODUCIBILITY ============
    RANDOM_STATE = 42
    TEST_SIZE = 0.2
    VAL_SIZE = 0.2
    CV_FOLDS = 5
    N_JOBS = -1
    
    # ============ DIRECTORIES ============
    MODEL_DIR = "models_nyc_taxi_mlflow"
    DATA_DIR = "data_nyc_taxi_mlflow"
    
    # ============ DATA FILTERING ============
    SAMPLE_SIZE = 200000
    MIN_TRIP_DURATION = 60
    MAX_TRIP_DURATION = 7200
    MIN_TRIP_DISTANCE = 0.1
    MAX_TRIP_DISTANCE = 50
    
    # ============ GEOGRAPHIC BOUNDS ============
    NYC_LAT_RANGE = (40.5, 40.9)
    NYC_LON_RANGE = (-74.3, -73.7)
    
    # ============ MODEL HYPERPARAMETERS ============
    RF_N_ESTIMATORS = 100
    RF_MAX_DEPTH = 20
    RF_MIN_SAMPLES_SPLIT = 10
    
    GB_N_ESTIMATORS = 100
    GB_LEARNING_RATE = 0.1
    GB_MAX_DEPTH = 5
    
    RIDGE_ALPHA = 10.0
    LASSO_ALPHA = 0.1
    ELASTIC_ALPHA = 0.1
    ELASTIC_L1_RATIO = 0.5
    
    # ============ MLflow CONFIG ============
    MLFLOW_EXPERIMENT_NAME = "nyc_taxi_production_mlflow"
    MLFLOW_MODEL_NAME = "nyc_taxi_predictor"
    MLFLOW_TRACKING_URI = f"sqlite:///{os.path.abspath('./mlflow_nyc_taxi.db')}"

config = Config()

# Create directories
for dir_path in [config.MODEL_DIR, config.DATA_DIR]:
    os.makedirs(dir_path, exist_ok=True)

# Set MLflow experiment with tags
experiment = mlflow.set_experiment(config.MLFLOW_EXPERIMENT_NAME)

# Update experiment tags
client.set_experiment_tag(experiment.experiment_id, "project", "nyc_taxi")
client.set_experiment_tag(experiment.experiment_id, "team", "data_science")
client.set_experiment_tag(experiment.experiment_id, "data_leakage", "none")
client.set_experiment_tag(experiment.experiment_id, "framework", "scikit-learn")

print("✅ MLflow Experiment Configured")
print(f"   Name: {config.MLFLOW_EXPERIMENT_NAME}")
print(f"   ID: {experiment.experiment_id}")
print(f"   Location: {experiment.artifact_location}")
print(f"\n📁 Model directory: {config.MODEL_DIR}")
print(f"🎲 Random state: {config.RANDOM_STATE}")

In [ ]:
# ============================================
# PART 2: Data Acquisition
# ============================================

import kagglehub

print("📥 Downloading NYC Yellow Taxi dataset...")
path = kagglehub.dataset_download("elemento/nyc-yellow-taxi-trip-data")
print(f"✅ Dataset path: {path}")

file = f"{path}/yellow_tripdata_2016-01.csv"

print("📊 Loading data in chunks...")
chunks = pd.read_csv(file, chunksize=500_000, low_memory=False)
df1 = next(chunks)
df2 = next(chunks)
df = pd.concat([df1, df2], ignore_index=True)

print(f"✅ Data loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

In [ ]:
# ============================================
# PART 3: Data Cleaning (No Leakage)
# ============================================

def clean_nyc_taxi_data(df, config):
    """Clean NYC taxi data with NO DATA LEAKAGE."""
    df_clean = df.copy()
    initial_rows = len(df_clean)
    
    # Calculate target
    df_clean['tpep_pickup_datetime'] = pd.to_datetime(df_clean['tpep_pickup_datetime'])
    df_clean['tpep_dropoff_datetime'] = pd.to_datetime(df_clean['tpep_dropoff_datetime'])
    df_clean['trip_duration_minutes'] = (
        df_clean['tpep_dropoff_datetime'] - df_clean['tpep_pickup_datetime']
    ).dt.total_seconds() / 60
    
    # Filter unrealistic values
    df_clean = df_clean[
        (df_clean['trip_duration_minutes'] >= config.MIN_TRIP_DURATION/60) &
        (df_clean['trip_duration_minutes'] <= config.MAX_TRIP_DURATION/60)
    ]
    df_clean = df_clean[
        (df_clean['trip_distance'] >= config.MIN_TRIP_DISTANCE) &
        (df_clean['trip_distance'] <= config.MAX_TRIP_DISTANCE)
    ]
    
    # Filter NYC coordinates
    df_clean = df_clean[
        df_clean['pickup_latitude'].between(*config.NYC_LAT_RANGE) &
        df_clean['pickup_longitude'].between(*config.NYC_LON_RANGE) &
        df_clean['dropoff_latitude'].between(*config.NYC_LAT_RANGE) &
        df_clean['dropoff_longitude'].between(*config.NYC_LON_RANGE)
    ]
    
    # Passenger count
    df_clean = df_clean[df_clean['passenger_count'].between(1, 6)]
    
    # Drop missing values
    df_clean = df_clean.dropna()
    
    # Sample if needed
    if config.SAMPLE_SIZE and len(df_clean) > config.SAMPLE_SIZE:
        df_clean = df_clean.sample(n=config.SAMPLE_SIZE, random_state=config.RANDOM_STATE)
    
    df_clean = df_clean.reset_index(drop=True)
    
    print(f"✅ Cleaned: {len(df_clean):,} rows ({len(df_clean)/initial_rows*100:.1f}% retained)")
    return df_clean

df_clean = clean_nyc_taxi_data(df, config)

In [ ]:
# ============================================
# PART 4: Define Features (NO DATA LEAKAGE)
# ============================================

PREDICTION_TIME_FEATURES = [
    'tpep_pickup_datetime',
    'pickup_longitude', 'pickup_latitude',
    'dropoff_longitude', 'dropoff_latitude',
    'passenger_count', 'VendorID', 'RatecodeID',
    'trip_distance', 'payment_type'
]

LEAKAGE_FEATURES = [
    'fare_amount', 'tip_amount', 'total_amount',
    'extra', 'mta_tax', 'tolls_amount',
    'improvement_surcharge', 'store_and_fwd_flag',
    'tpep_dropoff_datetime'
]

X = df_clean[PREDICTION_TIME_FEATURES]
y = df_clean['trip_duration_minutes'].values

print(f"✅ Features: {len(PREDICTION_TIME_FEATURES)} available at pickup")
print(f"🚫 Excluded: {len(LEAKAGE_FEATURES)} post-trip features")

In [ ]:
# ============================================
# PART 5: CRITICAL - Split BEFORE Feature Engineering
# ============================================

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=config.VAL_SIZE, random_state=config.RANDOM_STATE
)

split_info = {
    'train': len(X_train), 'train_pct': len(X_train)/len(X)*100,
    'val': len(X_val), 'val_pct': len(X_val)/len(X)*100,
    'test': len(X_test), 'test_pct': len(X_test)/len(X)*100
}

print("🔒 DATA SPLIT (Before Feature Engineering):")
print(f"   Train: {split_info['train']:,} ({split_info['train_pct']:.1f}%)")
print(f"   Val:   {split_info['val']:,} ({split_info['val_pct']:.1f}%)")
print(f"   Test:  {split_info['test']:,} ({split_info['test_pct']:.1f}%)")

In [ ]:
# ============================================
# PART 6: Feature Engineering Transformer
# ============================================

class NYCYellowTaxiFeatureEngineer(BaseEstimator, TransformerMixin):
    """Feature engineering with NO DATA LEAKAGE."""
    
    def __init__(self, config=None):
        self.config = config
        self.feature_names_ = []
    
    def fit(self, X, y=None):
        return self
    
    @staticmethod
    def haversine_distance(lat1, lon1, lat2, lon2):
        R = 3958.8
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        return 2 * R * np.arcsin(np.sqrt(a))
    
    def transform(self, X):
        X_df = X.copy()
        X_df['tpep_pickup_datetime'] = pd.to_datetime(X_df['tpep_pickup_datetime'])
        
        # Distance features
        X_df['haversine_distance'] = self.haversine_distance(
            X_df['pickup_latitude'], X_df['pickup_longitude'],
            X_df['dropoff_latitude'], X_df['dropoff_longitude']
        )
        
        delta_lat = X_df['dropoff_latitude'] - X_df['pickup_latitude']
        delta_lon = X_df['dropoff_longitude'] - X_df['pickup_longitude']
        X_df['manhattan_distance'] = np.abs(delta_lat) * 69.0 + np.abs(delta_lon) * 53.0
        X_df['direction_sin'] = np.sin(np.arctan2(delta_lat, delta_lon))
        X_df['direction_cos'] = np.cos(np.arctan2(delta_lat, delta_lon))
        
        # Temporal features
        X_df['pickup_hour'] = X_df['tpep_pickup_datetime'].dt.hour
        X_df['pickup_dayofweek'] = X_df['tpep_pickup_datetime'].dt.dayofweek
        X_df['pickup_month'] = X_df['tpep_pickup_datetime'].dt.month
        
        # Cyclical encoding
        X_df['hour_sin'] = np.sin(2 * np.pi * X_df['pickup_hour'] / 24)
        X_df['hour_cos'] = np.cos(2 * np.pi * X_df['pickup_hour'] / 24)
        X_df['dayofweek_sin'] = np.sin(2 * np.pi * X_df['pickup_dayofweek'] / 7)
        X_df['dayofweek_cos'] = np.cos(2 * np.pi * X_df['pickup_dayofweek'] / 7)
        
        # Time flags
        X_df['is_rush_hour'] = (
            (X_df['pickup_hour'].between(7, 9)) |
            (X_df['pickup_hour'].between(16, 18))
        ).astype(int)
        X_df['is_weekend'] = X_df['pickup_dayofweek'].isin([5, 6]).astype(int)
        
        # Airport features
        X_df['pickup_from_jfk'] = self.haversine_distance(
            X_df['pickup_latitude'], X_df['pickup_longitude'], 40.6413, -73.7781
        )
        X_df['pickup_from_lga'] = self.haversine_distance(
            X_df['pickup_latitude'], X_df['pickup_longitude'], 40.7769, -73.8740
        )
        X_df['is_jfk_trip'] = (X_df['pickup_from_jfk'] < 2).astype(int)
        X_df['is_lga_trip'] = (X_df['pickup_from_lga'] < 2).astype(int)
        
        # Efficiency metrics
        X_df['efficiency_ratio'] = X_df['haversine_distance'] / (X_df['trip_distance'] + 1e-8)
        X_df['distance_per_passenger'] = X_df['trip_distance'] / (X_df['passenger_count'] + 1e-8)
        
        # Categorical encoding
        X_df['is_vendor_2'] = (X_df['VendorID'] == 2).astype(int)
        X_df['is_credit_card'] = (X_df['payment_type'] == 1).astype(int)
        
        # Interaction features
        X_df['distance_times_passengers'] = X_df['trip_distance'] * X_df['passenger_count']
        X_df['haversine_times_hour'] = X_df['haversine_distance'] * X_df['pickup_hour']
        
        # Drop original columns
        cols_to_drop = ['tpep_pickup_datetime', 'VendorID', 'RatecodeID', 'payment_type']
        X_df = X_df.drop(columns=cols_to_drop, errors='ignore')
        
        # Select numeric columns
        numeric_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
        self.feature_names_ = numeric_cols
        
        return X_df[numeric_cols].values
    
    def get_feature_names(self):
        return self.feature_names_

In [ ]:
# ============================================
# PART 7: Outlier Handler (Fit on Training Only)
# ============================================

class OutlierHandler(BaseEstimator, TransformerMixin):
    """IQR-based outlier handling - fitted on training data only."""
    
    def __init__(self, factor=1.5):
        self.factor = factor
        self.lower_bounds_ = None
        self.upper_bounds_ = None
    
    def fit(self, X, y=None):
        self.lower_bounds_ = []
        self.upper_bounds_ = []
        for i in range(X.shape[1]):
            Q1 = np.percentile(X[:, i], 25)
            Q3 = np.percentile(X[:, i], 75)
            IQR = Q3 - Q1
            self.lower_bounds_.append(Q1 - self.factor * IQR)
            self.upper_bounds_.append(Q3 + self.factor * IQR)
        return self
    
    def transform(self, X):
        X_transformed = X.copy()
        for i in range(X.shape[1]):
            X_transformed[:, i] = np.clip(
                X_transformed[:, i],
                self.lower_bounds_[i],
                self.upper_bounds_[i]
            )
        return X_transformed

In [ ]:
# ============================================
# PART 8: Build Preprocessing Pipeline
# ============================================

preprocessor = Pipeline([
    ('feature_engineer', NYCYellowTaxiFeatureEngineer(config=config)),
    ('outlier_handler', OutlierHandler(factor=config.IQR_FACTOR)),
    ('scaler', RobustScaler())
])

# Fit on TRAINING only
preprocessor.fit(X_train)

# Transform all datasets
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.named_steps['feature_engineer'].get_feature_names()
print(f"✅ Engineered features: {len(feature_names)}")

In [ ]:
# ============================================
# PART 9: Define Model Portfolio
# ============================================

models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(random_state=config.RANDOM_STATE, alpha=config.RIDGE_ALPHA),
    'Lasso': Lasso(random_state=config.RANDOM_STATE, alpha=config.LASSO_ALPHA, max_iter=5000),
    'Random Forest': RandomForestRegressor(
        random_state=config.RANDOM_STATE,
        n_jobs=config.N_JOBS,
        n_estimators=config.RF_N_ESTIMATORS,
        max_depth=config.RF_MAX_DEPTH,
        min_samples_split=config.RF_MIN_SAMPLES_SPLIT
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        random_state=config.RANDOM_STATE,
        n_estimators=config.GB_N_ESTIMATORS,
        learning_rate=config.GB_LEARNING_RATE,
        max_depth=config.GB_MAX_DEPTH
    )
}

print(f"🎯 Model Portfolio: {len(models)} models")

In [ ]:
# ============================================
# PART 10: MLflow Training with Auto-Registration
# ============================================

def train_with_mlflow(model, X_train, y_train, X_val, y_val, model_name):
    """Train model with complete MLflow tracking and auto-registration."""
    
    with mlflow.start_run(run_name=model_name) as run:
        # Train model
        start_time = time.time()
        model.fit(X_train, y_train)
        training_time = time.time() - start_time
        
        # Predictions
        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)
        
        # Metrics
        metrics = {
            'train_r2': r2_score(y_train, y_train_pred),
            'val_r2': r2_score(y_val, y_val_pred),
            'train_rmse': np.sqrt(mean_squared_error(y_train, y_train_pred)),
            'val_rmse': np.sqrt(mean_squared_error(y_val, y_val_pred)),
            'train_mae': mean_absolute_error(y_train, y_train_pred),
            'val_mae': mean_absolute_error(y_val, y_val_pred),
            'training_time': training_time,
            'overfitting_gap': r2_score(y_train, y_train_pred) - r2_score(y_val, y_val_pred)
        }
        
        # Cross-validation
        cv_scores = cross_val_score(model, X_train, y_train, cv=3, scoring='r2')
        metrics['cv_r2_mean'] = cv_scores.mean()
        metrics['cv_r2_std'] = cv_scores.std()
        
        # Log parameters
        try:
            params = model.get_params()
            params = {k: str(v) if callable(v) else v for k, v in params.items()}
            mlflow.log_params(params)
        except:
            pass
        
        # Log metrics
        mlflow.log_metrics(metrics)
        
        # Log tags
        mlflow.set_tag('model_family', model_name)
        mlflow.set_tag('data_leakage', 'none')
        mlflow.set_tag('features_at_pickup', len(PREDICTION_TIME_FEATURES))
        mlflow.set_tag('engineered_features', len(feature_names))
        
        # Log dataset info
        mlflow.log_param('train_samples', X_train.shape[0])
        mlflow.log_param('val_samples', X_val.shape[0])
        mlflow.log_param('features', X_train.shape[1])
        
        # Infer signature and log model with AUTO-REGISTRATION
        signature = infer_signature(X_train, y_train_pred)
        
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path='model',
            signature=signature,
            registered_model_name=config.MLFLOW_MODEL_NAME  # ⚡ Auto-register!
        )
        
        return metrics, model, run.info.run_id


# Train all models with MLflow tracking
print("🚀 Training models with MLflow tracking & auto-registration...")
print("=" * 70)

results = {}
trained_models = {}
run_ids = {}

for name, model in models.items():
    print(f"\n📌 Training {name}...")
    try:
        metrics, trained_model, run_id = train_with_mlflow(
            model, X_train_processed, y_train, X_val_processed, y_val, name
        )
        results[name] = metrics
        trained_models[name] = trained_model
        run_ids[name] = run_id
        
        print(f"   ✓ Run ID: {run_id[:8]}")
        print(f"   ✓ Val R²: {metrics['val_r2']:.4f} | CV R²: {metrics['cv_r2_mean']:.4f}")
        print(f"   ✓ Val MAE: {metrics['val_mae']:.2f} min")
    except Exception as e:
        print(f"   ❌ Error: {str(e)[:100]}")

print("\n✅ All models trained and registered in MLflow!")

In [ ]:
# ============================================
# PART 11: Model Comparison & Best Model Selection
# ============================================

# Create comparison DataFrame
metrics_df = pd.DataFrame(results).T
metrics_df = metrics_df[['val_r2', 'val_mae', 'cv_r2_mean', 'training_time', 'overfitting_gap']]
metrics_df = metrics_df.sort_values('val_r2', ascending=False)

print("📊 MODEL COMPARISON (Best to Worst):")
print("=" * 80)
print(metrics_df.round(4).to_string())

# Identify best model
best_model_name = metrics_df.index[0]
best_model = trained_models[best_model_name]
best_run_id = run_ids[best_model_name]

print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   Run ID: {best_run_id[:8]}")
print(f"   Val R²: {results[best_model_name]['val_r2']:.4f}")
print(f"   Val MAE: {results[best_model_name]['val_mae']:.2f} minutes")

# Tag the best run
client.set_tag(best_run_id, "best_model", "true")
client.set_tag(best_run_id, "selection_criteria", "val_r2")

In [ ]:
# ============================================
# PART 12: Hyperparameter Tuning with MLflow
# ============================================

param_grids = {
    'Random Forest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [15, 20, 25, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'Gradient Boosting': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1, 0.15],
        'max_depth': [3, 4, 5],
        'subsample': [0.8, 0.9, 1.0]
    }
}

tuned_models = {}
tuned_run_ids = {}

for model_name in ['Random Forest', 'Gradient Boosting']:
    print(f"\n🔧 Tuning {model_name}...")
    
    with mlflow.start_run(run_name=f"{model_name}_tuned") as run:
        search = RandomizedSearchCV(
            models[model_name],
            param_grids[model_name],
            n_iter=10,
            cv=3,
            scoring='r2',
            n_jobs=config.N_JOBS,
            random_state=config.RANDOM_STATE
        )
        
        search.fit(X_train_processed, y_train)
        
        # Log tuning results
        mlflow.log_params(search.best_params_)
        mlflow.log_metric('best_cv_score', search.best_score_)
        mlflow.log_metric('improvement', 
                         search.best_score_ - results[model_name]['cv_r2_mean'])
        
        # Log best model with auto-registration
        signature = infer_signature(X_train_processed, search.predict(X_train_processed))
        mlflow.sklearn.log_model(
            sk_model=search.best_estimator_,
            artifact_path='tuned_model',
            signature=signature,
            registered_model_name=config.MLFLOW_MODEL_NAME
        )
        
        tuned_models[model_name] = search.best_estimator_
        tuned_run_ids[model_name] = run.info.run_id
        
        print(f"   ✓ Best CV R²: {search.best_score_:.4f}")
        print(f"   ✓ Improvement: +{search.best_score_ - results[model_name]['cv_r2_mean']:.4f}")
        print(f"   ✓ Best params: {search.best_params_}")

# Compare tuned vs untuned
if tuned_models:
    for name in tuned_models:
        y_val_pred_tuned = tuned_models[name].predict(X_val_processed)
        tuned_r2 = r2_score(y_val, y_val_pred_tuned)
        print(f"\n📊 {name} - Tuned Val R²: {tuned_r2:.4f} (vs {results[name]['val_r2']:.4f})")
    
    # Update best model if tuned model is better
    best_tuned_name = max(tuned_models.keys(), 
                         key=lambda x: r2_score(y_val, tuned_models[x].predict(X_val_processed)))
    best_tuned_r2 = r2_score(y_val, tuned_models[best_tuned_name].predict(X_val_processed))
    
    if best_tuned_r2 > results[best_model_name]['val_r2']:
        print(f"\n🏆 NEW BEST MODEL (Tuned): {best_tuned_name}")
        best_model_name = best_tuned_name
        best_model = tuned_models[best_tuned_name]
        best_run_id = tuned_run_ids[best_tuned_name]

In [ ]:
# ============================================
# PART 13: Final Test Evaluation
# ============================================

print("🔬 FINAL TEST SET EVALUATION")
print("=" * 60)

y_test_pred = best_model.predict(X_test_processed)

test_metrics = {
    'test_r2': r2_score(y_test, y_test_pred),
    'test_rmse': np.sqrt(mean_squared_error(y_test, y_test_pred)),
    'test_mae': mean_absolute_error(y_test, y_test_pred)
}

print(f"\n📊 Test Metrics:")
print(f"   • R² Score: {test_metrics['test_r2']:.4f} ({test_metrics['test_r2']*100:.1f}%)")
print(f"   • RMSE: {test_metrics['test_rmse']:.2f} minutes")
print(f"   • MAE: {test_metrics['test_mae']:.2f} minutes")

# Log final test metrics to the best run
with mlflow.start_run(run_id=best_run_id):
    mlflow.log_metrics(test_metrics)
    mlflow.set_tag('final_model', 'true')
    mlflow.set_tag('deployment_ready', 'true')

print(f"\n✅ Final metrics logged to run: {best_run_id[:8]}")

In [ ]:
# ============================================
# PART 14: Model Registry - Stage Transitions
# ============================================

print("🏛️ MLflow Model Registry - Stage Transitions")
print("=" * 60)

model_name = config.MLFLOW_MODEL_NAME

# Get all versions
try:
    all_versions = client.search_model_versions(f"name='{model_name}'")
    versions_sorted = sorted(all_versions, key=lambda x: int(x.version))
    
    print(f"\n📦 Model: {model_name}")
    print(f"   Total versions: {len(versions_sorted)}")
    
    # Find our best model version
    best_version = None
    for v in versions_sorted:
        if v.run_id == best_run_id:
            best_version = v.version
            print(f"   🏆 Best model version: {best_version} (Run ID: {best_run_id[:8]})")
            break
    
    if best_version:
        # Promote best model to Staging
        client.transition_model_version_stage(
            name=model_name,
            version=best_version,
            stage="Staging"
        )
        print(f"   ✅ Version {best_version} → Staging")
        
        # Add description
        client.update_model_version(
            name=model_name,
            version=best_version,
            description=f"Best model: {best_model_name} | Test R²: {test_metrics['test_r2']:.4f} | Test MAE: {test_metrics['test_mae']:.2f} min"
        )
        
        # Add tags
        client.set_model_version_tag(model_name, best_version, "algorithm", best_model_name)
        client.set_model_version_tag(model_name, best_version, "test_r2", str(test_metrics['test_r2']))
        client.set_model_version_tag(model_name, best_version, "test_mae", str(test_metrics['test_mae']))
        client.set_model_version_tag(model_name, best_version, "data_leakage", "none")
        
        # Archive older versions in Production
        production_versions = client.get_latest_versions(model_name, stages=["Production"])
        for prod_v in production_versions:
            if prod_v.version != best_version:
                client.transition_model_version_stage(
                    name=model_name,
                    version=prod_v.version,
                    stage="Archived"
                )
                print(f"   📦 Version {prod_v.version} → Archived")
        
        # Promote from Staging to Production (manual approval step)
        print("\n🔔 READY FOR PRODUCTION PROMOTION")
        print(f"   Run this cell to promote to Production:")
        print(f"""
client.transition_model_version_stage(
    name='{model_name}',
    version={best_version},
    stage='Production'
)
print(f"✅ Version {best_version} → Production")
""")
    else:
        print("❌ Best run not found in registry")
        
except Exception as e:
    print(f"❌ Error: {e}")

print("\n" + "=" * 60)
print("📊 Current Model Registry Status:")
for stage in ["None", "Staging", "Production", "Archived"]:
    try:
        versions = client.get_latest_versions(model_name, stages=[stage])
        if versions:
            v = versions[0]
            print(f"   • {stage:<10}: v{v.version} (Run: {v.run_id[:8]})")
    except:
        pass

In [ ]:
# ============================================
# PART 15: Promote to Production (Execute Manually)
# ============================================

# Uncomment to promote the staging model to production

"""
# Get the staging version
staging_versions = client.get_latest_versions(model_name, stages=["Staging"])
if staging_versions:
    staging_version = staging_versions[0].version
    
    # Archive current production
    prod_versions = client.get_latest_versions(model_name, stages=["Production"])
    for prod_v in prod_versions:
        client.transition_model_version_stage(
            name=model_name,
            version=prod_v.version,
            stage="Archived"
        )
    
    # Promote staging to production
    client.transition_model_version_stage(
        name=model_name,
        version=staging_version,
        stage="Production"
    )
    print(f"✅ Version {staging_version} promoted to Production")
"""

In [ ]:
# ============================================
# PART 16: Load Models from Registry
# ============================================

print("📥 Loading Models from MLflow Model Registry")
print("=" * 60)

# Method 1: Load by stage (Production)
try:
    model_prod = mlflow.sklearn.load_model(
        model_uri=f"models:/{model_name}/Production"
    )
    print(f"✅ Loaded Production model: {type(model_prod).__name__}")
except:
    print("⚠️ No Production model found")

# Method 2: Load by stage (Staging)
try:
    model_staging = mlflow.sklearn.load_model(
        model_uri=f"models:/{model_name}/Staging"
    )
    print(f"✅ Loaded Staging model: {type(model_staging).__name__}")
except:
    print("⚠️ No Staging model found")

# Method 3: Load by version
try:
    model_v1 = mlflow.sklearn.load_model(
        model_uri=f"models:/{model_name}/1"
    )
    print(f"✅ Loaded Version 1")
except:
    print("⚠️ Version 1 not found")

# Method 4: Load latest version
try:
    model_latest = mlflow.sklearn.load_model(
        model_uri=f"models:/{model_name}/latest"
    )
    print(f"✅ Loaded latest version")
except:
    print("⚠️ No latest version found")

# Test prediction with loaded model
if 'model_staging' in locals():
    sample = X_test_processed[:1]
    pred = model_staging.predict(sample)[0]
    print(f"\n📊 Test prediction: {pred:.2f} minutes")

In [ ]:
# ============================================
# PART 17: Save Local Deployment Package
# ============================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_version_local = f"nyc_taxi_{best_model_name.lower().replace(' ', '_')}_{timestamp}"
model_save_dir = os.path.join(config.MODEL_DIR, model_version_local)
os.makedirs(model_save_dir, exist_ok=True)

print(f"💾 Saving deployment package: {model_version_local}")
print("=" * 60)

# Save model and preprocessor
joblib.dump(best_model, os.path.join(model_save_dir, 'model.pkl'))
joblib.dump(preprocessor, os.path.join(model_save_dir, 'preprocessor.pkl'))
print(f"✅ Model and preprocessor saved")

# Save model card with MLflow metadata
model_card = {
    'model_name': best_model_name,
    'model_version': model_version_local,
    'mlflow_run_id': best_run_id,
    'mlflow_registered_model': model_name,
    'mlflow_experiment': config.MLFLOW_EXPERIMENT_NAME,
    'test_performance': test_metrics,
    'features': {
        'pickup_features': PREDICTION_TIME_FEATURES,
        'engineered_features': len(feature_names)
    },
    'deployment': {
        'mlflow_uri': f"models:/{model_name}/Staging",
        'local_path': model_save_dir,
        'timestamp': timestamp
    }
}

with open(os.path.join(model_save_dir, 'model_card.json'), 'w') as f:
    json.dump(model_card, f, indent=2)
print(f"✅ Model card saved")

print(f"\n📦 Package location: {model_save_dir}")
print(f"🔗 MLflow URI: models:/{model_name}/Staging")

In [ ]:
# ============================================
# PART 18: MLflow UI Instructions
# ============================================

print("🎯 MLflow UI Access Instructions")
print("=" * 60)
print(f"""
📁 DATABASE:
   {MLFLOW_DB_PATH}

🔴 COMMAND (Open NEW Terminal):
   cd {os.getcwd()}
   mlflow ui --backend-store-uri sqlite:///{MLFLOW_DB_PATH} --host 127.0.0.1 --port 5000

🌐 URL:
   http://127.0.0.1:5000

📸 WHAT YOU'LL SEE:
   • Experiment: '{config.MLFLOW_EXPERIMENT_NAME}'
   • {len(results)} model training runs with parameters & metrics
   • Registered Model: '{model_name}' with multiple versions
   • Stage transitions: Staging → Production → Archived
   • Model lineage: Run ID → Model Version → Stage
   • Artifacts: Model files, signatures, metadata

🏆 BEST MODEL:
   • Algorithm: {best_model_name}
   • Run ID: {best_run_id[:8]}
   • Version: {best_version if 'best_version' in locals() else 'N/A'}
   • Stage: Staging
   • Test R²: {test_metrics['test_r2']:.4f}
   • Test MAE: {test_metrics['test_mae']:.2f} minutes
""")

In [ ]:
# ============================================
# PART 19: Summary - MLflow Integration Complete
# ============================================

print("✅ MLFLOW INTEGRATION SUMMARY")
print("=" * 70)

mlflow_features = [
    "1. ✅ SQLITE BACKEND: Single file database for complete experiment tracking",
    f"   • Database: mlflow_nyc_taxi.db",
    f"   • Experiment: {config.MLFLOW_EXPERIMENT_NAME}",
    f"   • Registered Model: {model_name}",
    "",
    "2. ✅ EXPERIMENT TRACKING: All training runs fully logged",
    f"   • Models trained: {len(results)}",
    f"   • Best run ID: {best_run_id[:8]}",
    "   • Parameters: Model hyperparameters",
    "   • Metrics: R², RMSE, MAE, CV scores",
    "   • Tags: Data leakage status, feature counts",
    "",
    "3. ✅ MODEL REGISTRY: Automatic version management",
    f"   • Model name: {model_name}",
    f"   • Total versions: {len(all_versions) if 'all_versions' in locals() else 'N/A'}",
    "   • Auto-registration: mlflow.sklearn.log_model(registered_model_name=...)",
    "",
    "4. ✅ STAGE TRANSITIONS: Professional model lifecycle",
    "   • None → Staging → Production → Archived",
    f"   • Best model currently in: Staging",
    "   • Production ready: One-click promotion",
    "",
    "5. ✅ MODEL LINEAGE: Full traceability",
    "   • Run ID → Model Version → Stage",
    "   • Training code → Registered model → Deployment",
    "   • Complete audit trail in SQLite database",
    "",
    "6. ✅ DEPLOYMENT READY: Multiple loading methods",
    "   • models:/name/Production - Current production model",
    "   • models:/name/Staging - Candidate for production",
    "   • models:/name/1 - Specific version",
    "   • models:/name/latest - Most recent version",
]

for feature in mlflow_features:
    print(feature)

print("\n" + "=" * 70)
print("🎯 NEXT STEPS:")
print("=" * 70)
print(f"""
1. 📊 VIEW MLFLOW UI:
   mlflow ui --backend-store-uri sqlite:///{MLFLOW_DB_PATH} --host 127.0.0.1 --port 5000

2. 🚀 PROMOTE TO PRODUCTION:
   client.transition_model_version_stage(name='{model_name}', version={best_version if 'best_version' in locals() else 'X'}, stage='Production')

3. 📥 LOAD FOR INFERENCE:
   model = mlflow.sklearn.load_model(model_uri=f"models:/{model_name}/Production")

4. 📦 DEPLOYMENT PACKAGE:
   cd {model_save_dir}
   # Contains model.pkl, preprocessor.pkl, model_card.json

5. 🔄 RETRAINING:
   # All experiments already tracked - just run this notebook again!
   # New models will be auto-registered as new versions
""")

print("\n" + "=" * 70)
print("🎉 MLFLOW INTEGRATION COMPLETE - PRODUCTION READY!")
print("=" * 70)